# Ảnh hưởng của các tham số EHO

Mỗi cấu hình dùng cùng tập seed và báo cáo tất cả lần chạy. Mỗi bảng có fitness tốt nhất cuối, độ lệch chuẩn mẫu, thời gian và evaluations thực. Chế độ mặc định QUICK dùng 5 seed. Đặt MODE thành FULL để dùng 25 seed.

In [ ]:
import os
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from eho import EHOConfig
from objectives import rastrigin
from experiment_helpers.runner import (
    assert_reproducible, compare_configurations, results_frame, seeds_for,
)
from experiment_helpers.statistics import summarize_experiment_results

MODE = os.getenv("EHO_EXPERIMENT_MODE", "QUICK").upper()
seeds = seeds_for(MODE)
base = EHOConfig(population_size=20, max_generations=50)
print(MODE, seeds)
print(base)
assert_reproducible(rastrigin, base, seeds[0])
print("Kiểm tra tái lập cùng seed đã đạt.")

In [ ]:
from experiment_helpers.plotting import (
    plot_convergence, plot_fitness_comparison, plot_parameter_heatmap, plot_runtime_comparison,
)

## 1. Alpha

Khi alpha tăng, mỗi bước thường di chuyển xa hơn về pbest với cùng r. Chất lượng cuối có thể đổi theo cả hai chiều. Giữ mọi tham số khác cố định.

In [ ]:
configs = {value: replace(base, alpha=value) for value in (0.1, 0.3, 0.5, 0.7, 0.9)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
plot_convergence(groups)
plot_fitness_comparison(summary)
plt.show()
print(f"Trong mẫu này, alpha có mean thấp nhất là {summary['mean'].idxmin()}. Xem SD để đánh giá độ phân tán.")

## 2. Beta

Theo (12.2), beta nhỏ kéo hàng đầu đàn về gần gốc hơn. Beta bằng 0 đưa hàng này đúng về gốc, là cực tiểu đã biết của Rastrigin. Đặc điểm đó phụ thuộc bài toán.

In [ ]:
configs = {value: replace(base, beta=value) for value in (0.0, 0.1, 0.3, 0.5, 0.8)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
plot_convergence(groups)
plt.show()
print(f"Mean với beta = 0 là {summary.loc[0.0, 'mean']:.6g}. Mốc này kiểm tra cơ chế (12.2) trên Rastrigin.")

## 3A. Quần thể ở cùng số thế hệ

Giả thuyết là quần thể lớn có thêm cơ hội tìm nghiệm, đồng thời tốn thêm đánh giá. Tất cả chạy 50 thế hệ.

In [ ]:
configs = {size: replace(base, population_size=size) for size in (5, 10, 20, 40)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
plot_fitness_comparison(summary)
plot_runtime_comparison(summary)
plt.show()
print(f"Evaluations tăng từ {summary.evaluations.min():g} đến {summary.evaluations.max():g}. Cần so thêm ngân sách cố định.")

## 3B. Quần thể ở cùng ngân sách

Dùng đúng 1.200 evaluations gồm khởi tạo. Với P cá thể, số thế hệ là 1200/P − 1. Giả thuyết là thứ hạng có thể đổi khi số đánh giá được giữ cố định.

In [ ]:
configs = {size: replace(base, population_size=size, max_generations=1200 // size - 1)
           for size in (5, 10, 20, 40)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
assert frame.evaluations.eq(1200).all()
plot_convergence(groups, evaluations=True)
plot_fitness_comparison(summary)
plot_runtime_comparison(summary)
plt.show()
print(f"Khi cùng ngân sách, P có mean thấp nhất trong mẫu là {summary['mean'].idxmin()}.")

## 4. Số thế hệ

Kéo dài ngân sách ở cùng seed có thể giữ hoặc cải thiện best lịch sử. Đo mức cải thiện và chi phí; đoạn nằm ngang chỉ là quan sát trong ngân sách đã chạy.

In [ ]:
configs = {count: replace(base, max_generations=count) for count in (20, 50, 100, 200)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
plot_convergence(groups)
plot_runtime_comparison(summary)
plt.show()
print(f"Mean giảm {summary.loc[20, 'mean'] - summary.loc[200, 'mean']:.6g} từ 20 đến 200 thế hệ.")

## 5. Số chiều

Số biến và số hạng fitness tăng cùng dimension. So sánh ở cùng ngân sách, đọc fitness tổng và fitness trên mỗi chiều như hai mô tả khác nhau.

In [ ]:
configs = {dimension: EHOConfig(dimension=dimension, population_size=20, max_generations=50)
           for dimension in (2, 4, 8, 12)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
display(summary["mean"] / summary.index.to_numpy())
plot_fitness_comparison(summary)
plot_runtime_comparison(summary)
plt.show()
print(f"Dimension có độ phân tán mẫu lớn nhất: {summary['std'].idxmax()}.")

## 6. Tương tác Alpha và Beta

Giả thuyết là tác động của alpha phụ thuộc beta. Giữ P=20, d=4, G=50. Mỗi ô là mean fitness cuối trên toàn bộ seed.

In [ ]:
configs = {f"{alpha}/{beta}": replace(base, alpha=alpha, beta=beta)
           for beta in (0.1, 0.5) for alpha in (0.3, 0.5, 0.7)}
groups = compare_configurations(rastrigin, configs, seeds)
frame = results_frame(groups)
summary = summarize_experiment_results(frame)
display(summary)
heatmap = frame.pivot_table(index="beta", columns="alpha", values="fitness", aggfunc="mean")
plot_parameter_heatmap(heatmap)
plt.show()
print(f"Tổ hợp alpha/beta có mean thấp nhất trong lưới: {summary['mean'].idxmin()}, n={len(seeds)}.")

## Kết luận

Diễn giải alpha cùng mức beta đã thử. So sánh quần thể theo cùng số thế hệ và theo cùng số đánh giá trả lời hai câu hỏi khác nhau. Các bảng chỉ mô tả cấu hình, seed và ngân sách đã chạy.